In [1]:
!pip install -q "transformers==4.57.1" "peft==0.18.0" bitsandbytes accelerate "pandas==2.2.2" "pillow==11.3.0" tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.6 MB/s eta 0:00:00


In [2]:
import kagglehub
from pathlib import Path
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
COMP_DIR = Path(kagglehub.competition_download("pixels-to-predictions"))
print("Downloaded to:", COMP_DIR)

100%|██████████| 358M/358M [00:07<00:00, 52.6MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/competitions/pixels-to-predictions


In [5]:
import os, json, random, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import PeftModel

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS = "ABCDEFGHIJ"
BATCH_SIZE_INFER = 4

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
DATA_DIR = COMP_DIR / "data"
if not DATA_DIR.exists(): DATA_DIR = COMP_DIR

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

example_name = Path(val_df.iloc[0]["image_path"]).name
matches = list(COMP_DIR.rglob(example_name))
DATA_ROOT = matches[0].parents[2]
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 3109 | Val: 1048 | Test: 1008


In [7]:
def clean_text(x):
    if pd.isna(x): return ""
    return str(x).strip()

def get_candidate_token_ids(tokenizer):
    cids = {}
    for letter in CHOICE_LETTERS:
        forms = [letter, " "+letter, "\n"+letter, letter+".", " "+letter+"."]
        ids = set()
        for f in forms:
            enc = tokenizer.encode(f, add_special_tokens=False)
            if enc: ids.add(enc[-1])
        cids[letter] = sorted(ids)
    return cids

## Cell 5: Load v2

In [8]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="eager",
)
model = PeftModel.from_pretrained(base_model, "/content/drive/MyDrive/lora_v2_best")
model.eval()
candidate_ids = get_candidate_token_ids(processor.tokenizer)
print("Loaded v2 checkpoint")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Loaded v2 checkpoint


## Cell 6: Define all configs

In [9]:
def predict_with_config(df_batch, img_fn, prompt_fn):
    images = [img_fn(row) for _, row in df_batch.iterrows()]
    prompts = [prompt_fn(row) for _, row in df_batch.iterrows()]
    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}
    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]
    log_probs = torch.log_softmax(logits, dim=-1)
    preds = []
    for i, (_, row) in enumerate(df_batch.iterrows()):
        scores = [log_probs[i, candidate_ids[CHOICE_LETTERS[ci]]].max().item()
                  for ci in range(len(row["choices"]))]
        preds.append(int(np.argmax(scores)))
    return preds

# Image loaders
def img_224(row):
    return Image.open(DATA_ROOT / row["image_path"]).convert("RGB").resize((224,224), Image.BICUBIC)

def img_336(row):
    return Image.open(DATA_ROOT / row["image_path"]).convert("RGB").resize((336,336), Image.BICUBIC)

def img_native(row):
    return Image.open(DATA_ROOT / row["image_path"]).convert("RGB")

# Prompt variants
def prompt_full(row):
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))
    prompt = "<image>\nYou are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    if hint: prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"
    return prompt

def prompt_q_only(row):
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    prompt = "<image>\nYou are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"
    return prompt

def prompt_q_hint(row):
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    prompt = "<image>\nYou are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if hint: prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"
    return prompt

def prompt_q_lecture(row):
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    lecture = clean_text(row.get("lecture", ""))
    prompt = "<image>\nYou are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"
    return prompt

## Cell 7: Run all experiments

In [10]:
eval_df = val_df.sample(n=200, random_state=SEED).reset_index(drop=True)

experiments = [
    ("224 + full prompt", img_224, prompt_full),
    ("336 + full prompt", img_336, prompt_full),
    ("native + full prompt", img_native, prompt_full),
    ("224 + q only", img_224, prompt_q_only),
    ("224 + q+hint", img_224, prompt_q_hint),
    ("224 + q+lecture", img_224, prompt_q_lecture),
]

results = {}
for name, img_fn, prompt_fn in experiments:
    preds = []
    for start in range(0, len(eval_df), BATCH_SIZE_INFER):
        batch = eval_df.iloc[start:start+BATCH_SIZE_INFER]
        try:
            p = predict_with_config(batch, img_fn, prompt_fn)
            preds.extend(p)
        except RuntimeError:
            torch.cuda.empty_cache()
            for j in range(len(batch)):
                b = batch.iloc[j:j+1]
                p = predict_with_config(b, img_fn, prompt_fn)
                preds.extend(p)
                torch.cuda.empty_cache()

    acc = sum(p == a for p, a in zip(preds, eval_df["answer"])) / len(eval_df)
    results[name] = (preds, acc)
    print(f"  {name}: {acc:.4f}")
    torch.cuda.empty_cache()

print("\n=== Ranked Results ===")
for name, (_, acc) in sorted(results.items(), key=lambda x: -x[1][1]):
    marker = " ← best" if acc == max(v[1] for v in results.values()) else ""
    print(f"  {acc:.4f}  {name}{marker}")

  224 + full prompt: 0.6700
  336 + full prompt: 0.6550
  native + full prompt: 0.6750
  224 + q only: 0.5650
  224 + q+hint: 0.5950
  224 + q+lecture: 0.5850

=== Ranked Results ===
  0.6750  native + full prompt ← best
  0.6700  224 + full prompt
  0.6550  336 + full prompt
  0.5950  224 + q+hint
  0.5850  224 + q+lecture
  0.5650  224 + q only


## Cell 8: Submit the best

In [11]:
best_name = max(results, key=lambda k: results[k][1])
best_img_fn = dict(zip([e[0] for e in experiments], [e[1] for e in experiments]))[best_name]
best_prompt_fn = dict(zip([e[0] for e in experiments], [e[2] for e in experiments]))[best_name]
print(f"Submitting: {best_name} ({results[best_name][1]:.4f})")

all_preds = []
for start in tqdm(range(0, len(test_df), BATCH_SIZE_INFER), desc="Test"):
    batch = test_df.iloc[start:start+BATCH_SIZE_INFER]
    try:
        p = predict_with_config(batch, best_img_fn, best_prompt_fn)
        all_preds.extend(p)
    except RuntimeError:
        torch.cuda.empty_cache()
        for j in range(len(batch)):
            b = batch.iloc[j:j+1]
            p = predict_with_config(b, best_img_fn, best_prompt_fn)
            all_preds.extend(p)
            torch.cuda.empty_cache()

submission = pd.DataFrame({"id": test_df["id"], "answer": all_preds})
submission.to_csv("/content/submission.csv", index=False)
submission.to_csv("/content/drive/MyDrive/submission_ablation.csv", index=False)
print(f"Saved ({len(submission)} rows)")

from google.colab import files
files.download("/content/submission.csv")

Submitting: native + full prompt (0.6750)


Test:   0%|          | 0/252 [00:00<?, ?it/s]

Saved (1008 rows)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>